# Lesson 2.5 — Output Handling: Structured Output and Defensive Parsing

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Produce structured output via JSON prompting and tool_use schemas
- Validate output against a JSON schema before using it
- Parse defensively and recover from malformed output
- Treat confident model output with healthy skepticism

**You will produce:** A JSON-schema validator with a retry-on-invalid loop and defensive parsing.

## Concepts

Production code must not trust raw model text. **Output handling** patterns:

- **Structured output**: request JSON explicitly, or (more robustly) define a **tool** whose `input_schema` is your target shape and read the validated `tool_use` input. The schema both guides the model and documents the contract.
- **Response validation**: check the parsed output against a JSON schema (types, required keys, enums) before acting on it.
- **Defensive parsing**: wrap `json.loads` in try/except, strip code fences, and handle partial output; never assume a field exists.
- **Skepticism**: a confident-sounding answer can still be wrong - validate facts and ranges, and prefer re-asking over using bad data.

> **Reference:** [Anthropic - Tool use (structured output)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define a target schema + tool

A tool's input_schema doubles as your output contract.

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
        "confidence": {"type": "number"},
    },
    "required": ["sentiment", "confidence"],
}

TOOL = {"name": "record_sentiment", "description": "Record a sentiment label.",
        "input_schema": SCHEMA}
print("Tool contract ready:", TOOL['name'])

## Code-Along: Step 2 — Validate output against the schema

A tiny validator checks required keys, types, and enums.

In [ ]:
def validate(obj, schema):
    errors = []
    for key in schema.get("required", []):
        if key not in obj:
            errors.append(f"missing '{key}'")
    props = schema.get("properties", {})
    for key, val in obj.items():
        spec = props.get(key, {})
        if spec.get("type") == "number" and not isinstance(val, (int, float)):
            errors.append(f"'{key}' must be number")
        if spec.get("enum") and val not in spec["enum"]:
            errors.append(f"'{key}' not in {spec['enum']}")
    return errors

print(validate({"sentiment": "great", "confidence": "high"}, SCHEMA))

## Code-Along: Step 3 — Defensive parse + retry loop

Strip fences, try/except json.loads, and re-ask on invalid output.

In [ ]:
import json

def safe_parse(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, str(e)

def get_valid_sentiment(text, max_tries=2):
    for attempt in range(max_tries):
        raw = ('{"sentiment": "positive", "confidence": 0.9}'
               if not HAVE_API else _call_tool(text))
        obj, err = safe_parse(raw) if isinstance(raw, str) else (raw, None)
        if obj is None:
            continue
        if not validate(obj, SCHEMA):
            return obj
    return None  # give up safely rather than use bad data

def _call_tool(text):
    msg = client.messages.create(model=MODEL_HAIKU, max_tokens=200,
        tools=[TOOL], tool_choice={"type": "tool", "name": "record_sentiment"},
        messages=[{"role": "user", "content": text}])
    for block in msg.content:
        if block.type == "tool_use":
            return block.input
    return "{}"

print(get_valid_sentiment("I really enjoyed the show!"))

## Exercise

Extend `validate` to support a `'string'` type check and an `'integer'` type, then write a test that catches at least two schema violations in one object.

In [ ]:
def validate_plus(obj, schema):
    # TODO: start from validate(), add 'string' and 'integer' type checks
    # TODO: return the list of error strings
    pass

## Key Takeaways

- Prefer tool_use with an input_schema to get validated structured output.
- Validate parsed output against a schema (required keys, types, enums) before acting.
- Parse defensively: strip fences, try/except, and never assume fields exist.
- Stay skeptical of confident output - re-ask rather than use invalid data.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.